# Subquestion 4
Is the extension of the free student travel product to MBO students under 18 (1 January 2017) associated with an increase in train and bus/tram/metro use among 12–18 year olds, compared with 18–25 year olds?

In [81]:
# Loading packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import nbformat
idx = pd.IndexSlice

In [82]:
# Loading the datasets
path2 = '../Data_age_research/2_Mobility_per_person_by_transport_mode_and_purpose_(84710NED).csv'

dataset2 = pd.read_csv(path2, sep=";", decimal=',', skiprows=5, header=[0, 1, 2, 3], index_col=0, skipfooter=1, na_values=['.', ''], engine='python')

In [83]:
# Prepare the dataset

agegroups = ['12 to 18 years', '18 to 25 years']
transportmode = ['Train', 'Bus/tram/metro']

# Clean index years (remove the '*')
dataset2.index = dataset2.index.str.replace('*', '', regex=False).astype(int)

# Filter the data
df2_train = dataset2.loc[:, idx[agegroups, 'Train', 'Total', 'number']]
df2_btm = dataset2.loc[:, idx[agegroups, 'Bus/tram/metro', 'Total', 'number']]

# Simplify column names to age group level
df2_train.columns = df2_train.columns.get_level_values('Age')
df2_btm.columns = df2_btm.columns.get_level_values('Age')

# Calculate total public transport (PT) use
df2_total_pt = df2_train + df2_btm

# Index values relative to 2016 (2016 = 100%)
df2_indexed = (df2_total_pt / df2_total_pt.loc[2016]) * 100

In [86]:
# Create figure with 2 stacked subplots sharing the x-axis
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1,
    subplot_titles=(
        'Absolute Total PT Use (Train + Bus/Tram/Metro)', 
        'Relative Change vs. Reference Year 2016 (Index = 100%)'
    )
)

# Line colors
color_12_18 = 'blue'
color_18_25 = 'red'

# --- Subplot A: Absolute numbers ---
fig.add_trace(
    go.Scatter(
        x=df2_total_pt.index, 
        y=df2_total_pt['12 to 18 years'],
        mode='lines+markers',
        name='12–18 years',
        line=dict(color=color_12_18, width=2),
        marker=dict(symbol='circle', size=7),
        legendgroup='12_18'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df2_total_pt.index, 
        y=df2_total_pt['18 to 25 years'],
        mode='lines+markers',
        name='18–25 years',
        line=dict(color=color_18_25, width=2),
        marker=dict(symbol='square', size=7),
        legendgroup='18_25'
    ),
    row=1, col=1
)

# Policy change vertical line
fig.add_vline(
    x=2017, 
    line_dash="dash", 
    line_color="black",
    row=1,
    col=1, 
    annotation_text="Policy change (2017)", 
    annotation_position="bottom left")


# --- Subplot B: Indexed trend ---
fig.add_trace(
    go.Scatter(
        x=df2_indexed.index, 
        y=df2_indexed['12 to 18 years'],
        mode='lines+markers',
        name='12–18 years',
        line=dict(color=color_12_18, width=2),
        marker=dict(symbol='circle', size=7),
        legendgroup='12_18',
        showlegend=False
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=df2_indexed.index, 
        y=df2_indexed['18 to 25 years'],
        mode='lines+markers',
        name='18–25 years',
        line=dict(color=color_18_25, width=2),
        marker=dict(symbol='square', size=7),
        legendgroup='18_25',
        showlegend=False
    ),
    row=2, col=1
)

# Policy change vertical line with annotation for Subplot B
fig.add_vline(
    x=2017, line_dash="dash", line_color="black",
    annotation_text="Policy change (2017)",
    annotation_position="bottom left",
    row=2, col=1
)

# Add horizontal line for index in Subplot B
fig.add_hline(
    y=100, line_dash="dash", line_color="grey",
    annotation_text="Index 100", annotation_position="top right",
    row=2, col=1
)

# Axis labels and layout formatting
fig.update_yaxes(title_text="Trips p.p.p.d.", row=1, col=1)
fig.update_yaxes(title_text="Index (2016 = 100)", row=2, col=1)
fig.update_xaxes(title_text="Year", row=2, col=1)

fig.update_layout(
    height=750,
    width=900,
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)

fig.show()

In [94]:
# Calculations to get insight in the plots

# Categorize relevant periods
period_before = df2_total_pt.loc[2014:2016]        # Baseline (before policy)
period_short_term = df2_total_pt.loc[2017:2019]    # Short term effects (before COVID-19)
period_post_covid = df2_total_pt.loc[2022:2025]    # Post COVID effects

# Calculate mean values for each period
mean_before = period_before.mean()
mean_short = period_short_term.mean()
mean_post = period_post_covid.mean()

#  Calculate the procentual change compared to the baseline
pct_diff_short = ((mean_short - mean_before) / mean_before) * 100
pct_diff_post = ((mean_post - mean_before) / mean_before) * 100

# Creating overview table
overview = pd.DataFrame({
    'Avg 2014-2016 (Baseline)': mean_before,
    'Avg 2017-2019 (Short Term)': mean_short,
    'Avg 2022-2025 (Post-COVID)': mean_post,
    'Change Short Term (%)': pct_diff_short,
    'Change Post-COVID (%)': pct_diff_post
}).round(2)

display(overview)

,Avg 2014-2016 (Baseline),Avg 2017-2019 (Short Term),Avg 2022-2025 (Post-COVID),Change Short Term (%),Change Post-COVID (%)
Age,,,,,
12 to 18 years,0.24,0.25,0.21,4.11,-14.73
18 to 25 years,0.64,0.63,0.50,-1.05,-21.47
